# M4-09 Model Deployment

This notebook trains, serializes, tests, and reflects on deploying a Random Forest classifier for the Iris dataset.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## Task 1: Train and Serialize

The Iris dataset contains 150 flower samples with four measurements and three target species. I split the data into training and test sets so the model can be evaluated on unseen examples.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Feature names: {iris.feature_names}")
print(f"Target names: {iris.target_names}")

A Random Forest is a good baseline classifier for this small structured dataset because it performs well without much preprocessing.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

The trained model and the target labels are saved with `joblib` so the Flask app can load them without retraining.

In [ ]:
joblib.dump(model, "model.joblib")
joblib.dump(iris.target_names, "target_names.joblib")

print("Saved model.joblib")
print("Saved target_names.joblib")

After serialization, I load the model back and verify that it produces identical predictions on the same test set. This confirms that the saved artifact preserves the trained model behavior.

In [ ]:
loaded_model = joblib.load("model.joblib")
loaded_predictions = loaded_model.predict(X_test)

print("Predictions identical after reload:", np.array_equal(y_pred, loaded_predictions))

## Task 3: Test the Flask API

Before running the next cells, start the Flask server in a terminal with `python app.py`. The API should run at `http://localhost:5000`.

In [ ]:
BASE_URL = "http://localhost:5000"

### Health Check

The health endpoint confirms that the service is running and reachable.

In [ ]:
health_response = requests.get(f"{BASE_URL}/health")
print("Status code:", health_response.status_code)
print("Response:", health_response.json())

### Single Prediction

This request sends one valid Iris sample and returns a human-readable species name plus class probabilities.

In [ ]:
sample = [5.1, 3.5, 1.4, 0.2]
predict_response = requests.post(f"{BASE_URL}/predict", json={"features": sample})
print("Status code:", predict_response.status_code)
print("Response:", predict_response.json())

### Error Handling

The API should reject malformed inputs with status code 400 and a descriptive error message.

In [ ]:
malformed_requests = {
    "Missing features key": {},
    "Wrong number of features": {"features": [5.1, 3.5, 1.4]},
    "Non-numeric values": {"features": ["a", "b", "c", "d"]},
}

for test_name, payload in malformed_requests.items():
    response = requests.post(f"{BASE_URL}/predict", json=payload)
    print(f"{test_name}:")
    print("  Status code:", response.status_code)
    print("  Response:", response.json())

### Batch Prediction

The batch endpoint accepts multiple samples in one request. I compare the API predictions with local model predictions to verify consistency.

In [ ]:
batch_samples = X_test[:5].tolist()
batch_response = requests.post(f"{BASE_URL}/predict_batch", json={"samples": batch_samples})
batch_json = batch_response.json()

api_predictions = [item["predicted_class"] for item in batch_json["predictions"]]
local_predictions = [iris.target_names[index] for index in loaded_model.predict(X_test[:5])]

print("Status code:", batch_response.status_code)
print("API predictions:", api_predictions)
print("Local predictions:", local_predictions)
print("Predictions match:", api_predictions == local_predictions)

## Task 4: Reflection

### Production deployment

For production, I would not use Flask's development server. I would serve the app with a WSGI server such as Gunicorn or uWSGI, run it inside a Docker container, configure paths and runtime settings with environment variables, add structured logging, and place the service behind HTTPS using a reverse proxy or cloud load balancer.

### Model versioning

I would store every trained model with a version number, training date, dataset version, metrics, and configuration. A new model should only be promoted if it beats the current production model on agreed evaluation metrics. If a retrained model performs worse, it should remain in staging or be rolled back automatically.

### Monitoring

I would monitor request volume, prediction latency, error rates, input validation failures, model confidence, and input feature drift. When true labels become available, I would also monitor live accuracy and class-level performance.

### Scaling to 1,000 requests per second

At higher traffic, I would run multiple API workers behind a load balancer, use autoscaling containers or Kubernetes, keep the model loaded in memory per worker, benchmark CPU and memory usage, and consider batching or a dedicated model-serving platform if latency requirements were strict.